# Model Comparison: Logistic Regression, Random Forest, and Decision Tree

This notebook loads the credit card approval dataset, trains three models, and compares their performance. It also runs hyperparameter tuning for each model and reports the best parameters and test set metrics.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
repo_root = Path.cwd()
for parent in [repo_root] + list(repo_root.parents):
    candidate = parent / 'Epic 1 - Data Collection' / 'creditcard.csv'
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError('creditcard.csv not found in Epic 1 - Data Collection')

print('Loading dataset from', data_path)
df = pd.read_csv(data_path)
print(df.shape)
print(df.columns.tolist())

In [ ]:
target = 'Class' if 'Class' in df.columns else df.columns[-1]   
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if len(y.unique()) > 1 else None
)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Target distribution:')
print(y.value_counts(normalize=True))

## 1. Baseline Models
Train baseline Logistic Regression, Random Forest, and Decision Tree models using default settings.

In [ ]:
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
}

results = {}
for name, model in models.items():
    print(f'
Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
    }
    print(f'{name} accuracy: {results[name][accuracy]:.4f}')

In [ ]:
for name, outcome in results.items():
    print('
' + '=' * 60)
    print(name)
    print('Accuracy:', f'{outcome['accuracy']:.4f}')
    print('Classification Report:
', outcome['report'])
    print('Confusion Matrix:
', outcome['confusion_matrix'])

## 2. Hyperparameter Tuning
Use `GridSearchCV` to find better hyperparameters for each model.

In [ ]:
param_grids = {
    'Logistic Regression': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__solver': ['liblinear', 'lbfgs'],
    },
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
    },
    'Decision Tree': {
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5, 10],
    },
}
tuned_results = {}
search_order = ['Logistic Regression', 'Random Forest', 'Decision Tree']
for name in search_order:
    print(f'
Tuning {name}...')
    if name == 'Logistic Regression':
        base = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))])
    elif name == 'Random Forest':
        base = RandomForestClassifier(random_state=42, n_jobs=-1)
    else:
        base = DecisionTreeClassifier(random_state=42)

    grid = GridSearchCV(base, param_grids[name], cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)
    tuned_results[name] = {
        'best_params': grid.best_params_,
        'best_score': grid.best_score_,
        'accuracy': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'model': grid.best_estimator_,
    }
    print(f'Best params for {name}:', grid.best_params_)
    print(f'Test accuracy: {tuned_results[name]['accuracy']:.4f}')

In [ ]:
comparison = []
for name, outcome in tuned_results.items():
    comparison.append({
        'model': name,
        'test_accuracy': outcome['accuracy'],
        'cv_best_score': outcome['best_score'],
        'best_params': outcome['best_params'],
    })
comparison_df = pd.DataFrame(comparison)
comparison_df

## 3. Save Comparison Results
Save the tuned comparison results and predictions to the Epic 4 folder.

In [ ]:
output_dir = Path('project folder/Epic 4 - Model Building')
output_dir.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(output_dir / 'model_comparison_summary.csv', index=False)
print('Saved model comparison summary to', output_dir / 'model_comparison_summary.csv')

for name, outcome in tuned_results.items():
    report_path = output_dir / f'{name.replace(' ', '_').lower()}_report.txt'
    with open(report_path, 'w') as fh:
        fh.write(outcome['report'])
    print('Saved report for', name, 'to', report_path)